# Практическое задание: анализ датчиков из ROS `.bag`

**Выполнила:** Холина Елена

**Данные:** запись с робота Unitree A1 (ИТ-Центр МАИ), файл `20230117_183740.bag`.

**Что сдаём:** только блок *«Практическое задание»* — считать параметры датчиков (акселерометр) и построить графики в `matplotlib`. Учебный разбор топиков/камер в этот файл **не копируем**.

## Подготовка

1. Загрузите `20230117_183740.bag` в Colab (иконка «Файлы» → загрузка) или положите рядом с ноутбуком локально.
2. Установите `bagpy` (в Colab может занять несколько минут).

In [ ]:
# bagpy тянет jinja2 3.0.x — предупреждение про flask можно игнорировать для этой лабы
%pip install -q bagpy

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bagpy import bagreader

BAG_NAME = "20230117_183740.bag"

def find_bag_file() -> Path:
    roots = []
    if os.environ.get("COLAB_RELEASE_TAG"):
        roots.append(Path("/content"))
    roots.extend([
        Path.cwd(),
        Path(r"C:/Users/kholina/Desktop/bagpy_DZ_Холина"),
        Path(r"C:/Users/kholina/Downloads"),
        Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd(),
    ])
    seen = set()
    for root in roots:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        direct = root / BAG_NAME
        if direct.is_file() and direct.stat().st_size > 0:
            return direct
        for hit in root.rglob("*.bag"):
            if hit.is_file() and hit.stat().st_size > 0:
                print("Найден bag:", hit)
                return hit
    return Path(BAG_NAME)  # для сообщения об ошибке

bag_path = find_bag_file()
print("Ищем файл:", BAG_NAME)
print("Текущая папка (cwd):", Path.cwd().resolve())

if not bag_path.is_file() or bag_path.stat().st_size == 0:
    print("\n--- В папке ноутбука нет .bag. Содержимое cwd ---")
    for item in sorted(Path.cwd().iterdir())[:30]:
        print(" ", item.name, "(папка)" if item.is_dir() else f"({item.stat().st_size} байт)")
    raise FileNotFoundError(
        "Положите 20230117_183740.bag в папку bagpy_DZ_Холина "
        "(рядом с bagpy_DZ_Холина.ipynb) и перезапустите ячейку."
    )

print("OK, bag:", bag_path.resolve())
print("Размер, МБ:", round(bag_path.stat().st_size / 1024 / 1024, 1))

b = bagreader(str(bag_path))
print("Топиков:", len(b.topic_table))

## Практическое задание

* Попробуйте считать какие-либо параметры датчиков из данного файла, например, акселерометра
* Постройте параметры в виде графиков в matplotlib

---

**Мой выбор:** топик IMU акселерометра `/device_0/sensor_2/Accel_0/imu/data` — линейное ускорение и угловая скорость (стандартные поля `sensor_msgs/Imu`).

In [ ]:
TOPIC_ACCEL = "/device_0/sensor_2/Accel_0/imu/data"
csv_path = b.message_by_topic(TOPIC_ACCEL)
df = pd.read_csv(csv_path)

print("Строк:", len(df), "| Столбцов:", len(df.columns))
print("Время, с:", df["Time"].min(), "—", df["Time"].max())
df[["Time", "linear_acceleration.x", "linear_acceleration.y", "linear_acceleration.z",
    "angular_velocity.x", "angular_velocity.y", "angular_velocity.z"]].head()

In [ ]:
# Производные параметры (свои, не только одна ось Y как в демо-примере)
df = df.sort_values("Time").reset_index(drop=True)

df["accel_mag"] = np.sqrt(
    df["linear_acceleration.x"] ** 2
    + df["linear_acceleration.y"] ** 2
    + df["linear_acceleration.z"] ** 2
)
df["gyro_mag"] = np.sqrt(
    df["angular_velocity.x"] ** 2
    + df["angular_velocity.y"] ** 2
    + df["angular_velocity.z"] ** 2
)

print(df[["accel_mag", "gyro_mag"]].describe().round(4))

In [ ]:
# График 1: три оси линейного ускорения во времени
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["Time"], df["linear_acceleration.x"], label="ax", lw=1)
ax.plot(df["Time"], df["linear_acceleration.y"], label="ay", lw=1)
ax.plot(df["Time"], df["linear_acceleration.z"], label="az", lw=1)
ax.set_title("Линейное ускорение (акселерометр), м/с²")
ax.set_xlabel("Время, с")
ax.set_ylabel("Ускорение")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# График 2: модуль ускорения и модуля гироскопа
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df["Time"], df["accel_mag"], color="#c0392b", lw=1.2)
axes[0].set_ylabel("|a|, м/с²")
axes[0].set_title("Модуль линейного ускорения")
axes[0].grid(alpha=0.3)

axes[1].plot(df["Time"], df["gyro_mag"], color="#2980b9", lw=1.2)
axes[1].set_xlabel("Время, с")
axes[1].set_ylabel("|ω|, рад/с")
axes[1].set_title("Модуль угловой скорости (гироскоп)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# График 3: распределение |a| (гистограмма)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["accel_mag"], bins=40, color="#27ae60", edgecolor="white", alpha=0.85)
ax.set_xlabel("|a|, м/с²")
ax.set_ylabel("Число измерений")
ax.set_title("Распределение модуля ускорения")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# График 4: scatter ay vs az (другой вид, не копия scatter Time–ay из примера)
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(
    df["linear_acceleration.y"],
    df["linear_acceleration.z"],
    c=df["Time"],
    s=12,
    cmap="viridis",
    alpha=0.6,
)
ax.set_xlabel("ay, м/с²")
ax.set_ylabel("az, м/с²")
ax.set_title("Фазовая траектория: ay vs az (цвет = время)")
plt.colorbar(sc, ax=ax, label="Time, s")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Выводы

1. Из `.bag` успешно извлечён поток IMU акселерометра; около **26 с** записи, тысячи сообщений.
2. Линейное ускорение по осям колеблется вокруг значений близких к **гравитации + вибрации** корпуса (робот не перемещали).
3. Модуль |a| и |ω| удобнее для анализа, чем одна ось: видна общая «энергия» вибраций.
4. Гистограмма |a| показывает, что большинство значений сосредоточено в узком диапазоне — типично для статического положения с шумом датчика.

---
*Демо-часть ноутбука (таблица топиков, камеры) — по шаблону курса; этот файл — только индивидуальная часть ДЗ.*